# Deriving Virtual Memory and Multi-level Page Tables from First Principles

## Part 1: Why Do We Need Virtual Memory?

If we want each Process to believe it has its own independent Address Space, what mechanism would we need to introduce?

Why isn't Software alone sufficient? Why do we need Hardware support?

If the CPU uses Virtual Addresses (VA), how does the Hardware determine the corresponding Physical Addresses (PA)?
It needs the mapping!

## Part 2: How Should We Design VA → PA Mapping?

- If we take the most intuitive approach and simply store a large number of `(VA, PA)` pairs, what problems would arise?

Storing is easy but looking for mapping needs to iterate through, too slow.


- If we want to avoid searching through these pairs one by one, could we use the VA directly as an Array Index?

yes, much better. But it takes lot of space, 2^27

- If the Page Size is 4KB, how many bits of a VA do not need to participate in Page Translation?

offset 12 bits can be skipped. It come from va. why 12 bits? 2^12 = 4kb page.


## Part 3: How Can We Design a Page Table Entry (PTE) Ourselves?

- If we have decided to establish Mappings at Page granularity, what information must a PTE store at a minimum?

physical address + permission info

- Does a PTE need to store the complete Physical Address? Why or why not?

don't need to store the 12 bits offset.

- Besides the PPN, what control information must a PTE store? Why?

can read, write, execute etc, lets give such 10 bits.

- Normally how many bits for physical address?

32 bits for 32bit-cpu, 46-52 bits for modern cpu. Such bits number is more than enough for RAM. Ex. 16G ram only needs 34 bits.

physical address (PA) bits needs to consider future RAM increase, and it will also affect the size of PTE! Since PTE needs to store PA, the more bits it needs, the more PTE stores!


- If a PTE requires at least 54 bits, should we choose 7 bytes or 8 bytes? Why?

Both works, but we want to fill up a 4kb page with PTE, no leftover space. So that index calculation and other things becomes much easier!


## Part 4: Why Should One Page Table Occupy Exactly One Physical Page?

- Where is the Page Table itself stored?

It's stored in physical RAM.

- When the MMU looks up a Page Table, does it use Virtual Addresses or Physical Addresses? Why?

It needs physical address. MMU translates va -> pa, Page Table is **source of truth** mapping, need to from physical address.


- If the OS's `kalloc()` can only allocate one 4KB Page at a time, why is it convenient for a Page Table to occupy exactly 4KB?

free, alloc, physical continuity all becomes a lot easier. If a page table is larger or smaller than 4kb, either waste space or code logic becomes much more complicated.


## Part 5: Why Isn't a Single-level Page Table Enough?

- If a Page Table is 4KB and each PTE is 8 bytes, how many address it can map to?

4k / 8 = 512 entries, each is 4kb, so 2Mb.

That's why use multi-level page table. Each level page table entry store the physical address of the next level page table PA. Eventually, the final PTE store the physical page address to va + permissions.

- Why VA spends 9 bits for each level page table?

Those bits are index! hardware knows the root page table's PA, it has 512 entries = 2^9, so use the first 9 bits as index -> entry -> PA of the next level page table. 

Only the last level page table stores permission, using all 10 bits. Using this to tell MMU whether the PTE points to the final PA or the PA of the next page table. 

If RWX is all 0, then it's not leaf! Otherwise it's meaningless!